# 🔬 KLA Hackathon — NAFNet-SR Image Restoration
**AI-Based Restoration of Degraded Semiconductor Inspection Images**

### Before running:
1. Go to `Runtime → Change runtime type → T4 GPU` → Save
2. Upload `train.zip` and `test_noisyLR.zip` to a folder called **`kla_data`** in your Google Drive root
3. Then click `Runtime → Run all` — everything is pre-configured!

**Expected training time:** ~2h (T4 GPU) | ~45min (A100)

In [ ]:
# ─────────────────────────────────────────────
# CELL 1: Check GPU
# ─────────────────────────────────────────────
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ─────────────────────────────────────────────
# CELL 2: Mount Google Drive
# ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
# ─────────────────────────────────────────────
# CELL 3: Clone repo & install dependencies
# ─────────────────────────────────────────────
import os

# Pre-configured — no edits needed!
REPO_URL = 'https://github.com/norriy0u/kla-image-restoration.git'
REPO_DIR = '/content/kla_restoration'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!pip install -r requirements.txt -q
print('✓ Setup complete!')

In [ ]:
# ─────────────────────────────────────────────
# CELL 4: Extract dataset from Google Drive
# ─────────────────────────────────────────────
# Expects in My Drive/kla_data/:
#   train.zip  and  test_noisyLR.zip
import os

DRIVE_DATA = '/content/drive/MyDrive/kla_data'
LOCAL_DATA = '/content/kla_data'

# Check if already extracted (skips unzip on re-runs)
if not os.path.exists(f'{LOCAL_DATA}/train/train/GT'):
    print('Extracting train.zip ...')
    !unzip -q {DRIVE_DATA}/train.zip -d {LOCAL_DATA}/
    print('Extracting test_noisyLR.zip ...')
    !unzip -q {DRIVE_DATA}/test_noisyLR.zip -d {LOCAL_DATA}/
    print('✓ Extraction complete!')
else:
    print('✓ Data already extracted — skipping.')

# Dataset paths (confirmed from actual data structure)
GT_DIR   = f'{LOCAL_DATA}/train/train/GT'
LR_DIR   = f'{LOCAL_DATA}/train/train/NoisyLR'
TEST_DIR = f'{LOCAL_DATA}/test_noisyLR/NoisyLR'

# Verify
gt_count = len(list(__import__('pathlib').Path(GT_DIR).glob('*.npy')))
lr_count = len(list(__import__('pathlib').Path(LR_DIR).glob('*.npy')))
test_count = len(list(__import__('pathlib').Path(TEST_DIR).glob('*.npy')))
print(f'GT files:    {gt_count}')
print(f'NoisyLR:     {lr_count}')
print(f'Test files:  {test_count}')
if gt_count == 3200 and lr_count == 3200:
    print('✓ All 3200 training pairs found!')
else:
    print('⚠️  File count mismatch — check Drive paths!')

In [ ]:
# ─────────────────────────────────────────────
# CELL 5: Quick data inspection
# ─────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import os, glob

gt_files = sorted(glob.glob(os.path.join(GT_DIR, '*.npy')))
lr_files = sorted(glob.glob(os.path.join(LR_DIR, '*.npy')))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    idx = i * 800
    gt = np.load(gt_files[min(idx, len(gt_files)-1)])
    lr = np.load(lr_files[min(idx, len(lr_files)-1)])
    lr_vis = np.clip(lr, 0, 1)

    axes[0][i].imshow(gt, cmap='gray', vmin=0, vmax=1)
    axes[0][i].set_title(f'GT #{min(idx, len(gt_files)-1)} (256×256)\nrange=[{gt.min():.2f},{gt.max():.2f}]', fontsize=9)
    axes[0][i].axis('off')

    axes[1][i].imshow(lr_vis, cmap='gray', vmin=0, vmax=1)
    axes[1][i].set_title(f'NoisyLR #{min(idx, len(lr_files)-1)} (128×128)\nmax={lr.max():.2f} (speckle overflow={lr.max()>1:.0f})', fontsize=9)
    axes[1][i].axis('off')

plt.suptitle('Sample Training Pairs — Top: GT (256×256), Bottom: NoisyLR (128×128)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/sample_pairs.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: /content/sample_pairs.png')

In [ ]:
# ─────────────────────────────────────────────
# CELL 6: Train! (~2h on T4, ~45min on A100)
# ─────────────────────────────────────────────
# Batch size: 16 for A100, 8 for T4 (safe)
BATCH_SIZE = 8
EPOCHS = 200

!python train.py \
    --gt_dir {GT_DIR} \
    --lr_dir {LR_DIR} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers 2 \
    --val_fraction 0.1 \
    --patch_size_gt 256 \
    --model_variant base \
    --weights_dir ./weights \
    --log_dir ./logs

In [ ]:
# ─────────────────────────────────────────────
# CELL 7: Launch TensorBoard (open in side panel)
# ─────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir ./logs

In [ ]:
# ─────────────────────────────────────────────
# CELL 8: Evaluate on validation set (PSNR/SSIM)
# ─────────────────────────────────────────────
import os, sys
sys.path.insert(0, '.')
os.makedirs('/content/val_outputs', exist_ok=True)

!python evaluate.py \
    --input_dir {LR_DIR} \
    --output_dir /content/val_outputs \
    --gt_dir {GT_DIR} \
    --weights ./weights/best_model.pt \
    --batch_size 8

import json
with open('/content/val_outputs/metrics.json') as f:
    m = json.load(f)
print('\n=== VALIDATION METRICS ===')
for k, v in m.items():
    print(f'  {k}: {v}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 9: Run inference on official test set
# ─────────────────────────────────────────────
import os
os.makedirs('/content/test_outputs', exist_ok=True)

!python evaluate.py \
    --input_dir {TEST_DIR} \
    --output_dir /content/test_outputs \
    --weights ./weights/best_model.pt \
    --batch_size 8

import json
with open('/content/test_outputs/metrics.json') as f:
    print('=== TEST INFERENCE STATS ===')
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# ─────────────────────────────────────────────
# CELL 10: Visualise Before → After → GT
# ─────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import glob, os

gt_files  = sorted(glob.glob(os.path.join(GT_DIR, '*.npy')))
lr_files  = sorted(glob.glob(os.path.join(LR_DIR, '*.npy')))

# Pick 4 samples with heaviest speckle (max pixel > 1.0)
lr_maxvals = sorted([(np.load(f).max(), f) for f in lr_files[:300]], reverse=True)
sample_files = [f for _, f in lr_maxvals[:4]]

fig, axes = plt.subplots(4, 3, figsize=(15, 20))
col_titles = ['NoisyLR Input (128×128)', 'NAFNet-SR Output (256×256)', 'Ground Truth (256×256)']
col_colors = ['#e74c3c', '#2ecc71', '#3498db']

for row, lr_path in enumerate(sample_files):
    stem    = os.path.splitext(os.path.basename(lr_path))[0]
    gt_path = os.path.join(GT_DIR, f'{stem}.npy')
    out_npy = f'/content/val_outputs/{stem}.npy'

    lr_arr  = np.load(lr_path)
    gt_arr  = np.load(gt_path) if os.path.exists(gt_path) else np.zeros((256,256))
    out_arr = np.load(out_npy)  if os.path.exists(out_npy)  else np.zeros((256,256))

    for col, (arr, title, color) in enumerate(zip([lr_arr, out_arr, gt_arr], col_titles, col_colors)):
        axes[row][col].imshow(np.clip(arr, 0, 1), cmap='gray', vmin=0, vmax=1)
        axes[row][col].set_title(f'{title}\n[{arr.min():.2f}, {arr.max():.2f}]',
                                  fontsize=10, color=color, fontweight='bold')
        axes[row][col].axis('off')
    axes[row][0].set_ylabel(f'Sample {stem}', fontsize=10, rotation=90, labelpad=15)

plt.suptitle('NAFNet-SR: Degraded Input → Restored → Ground Truth', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/before_after_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: /content/before_after_comparison.png')
print('Use this image in Slide 6 of your PPT!')

In [ ]:
# ─────────────────────────────────────────────
# CELL 11: Save everything to Google Drive
# ─────────────────────────────────────────────
import shutil, os

DRIVE_OUT = '/content/drive/MyDrive/kla_submission'
os.makedirs(DRIVE_OUT, exist_ok=True)

# Model weights
shutil.copy('./weights/best_model.pt', f'{DRIVE_OUT}/best_model.pt')
print('✓ Saved weights to Drive')

# Test outputs (submission)
shutil.copytree('/content/test_outputs', f'{DRIVE_OUT}/test_outputs', dirs_exist_ok=True)
print('✓ Saved test outputs to Drive')

# Before/after comparison image for PPT
shutil.copy('/content/before_after_comparison.png', f'{DRIVE_OUT}/before_after_comparison.png')
shutil.copy('/content/sample_pairs.png',            f'{DRIVE_OUT}/sample_pairs.png')
print('✓ Saved PPT images to Drive')

print(f'\n📁 All saved to: {DRIVE_OUT}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 12: [OPTIONAL] TTA inference — best quality
# ─────────────────────────────────────────────
# 4x slower but ~0.3 dB better PSNR
import os
os.makedirs('/content/test_outputs_tta', exist_ok=True)

!python evaluate.py \
    --input_dir {TEST_DIR} \
    --output_dir /content/test_outputs_tta \
    --weights ./weights/best_model.pt \
    --tta \
    --batch_size 1

import json
with open('/content/test_outputs_tta/metrics.json') as f:
    print('=== TTA Results ===')
    print(json.dumps(json.load(f), indent=2))